# 👁️ Diabetic Retinopathy Detection with InceptionV3
### End-to-End Machine Learning Pipeline

This notebook demonstrates:
1. **Dataset Loading & Preprocessing**: Ben Graham contrast enhancement and circular ROI cropping.
2. **Exploratory Data Analysis (EDA)**: Class distributions and retinal lesion characteristics.
3. **InceptionV3 Transfer Learning**: Model construction, feature extraction, and fine-tuning.
4. **Evaluation**: Confusion matrix, Quadratic Weighted Kappa (QWK), and Multi-class ROC curves.
5. **Explainability**: Grad-CAM attention heatmap generation.

In [ ]:
import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Ensure project root is in sys.path
sys.path.append('..')
from src.config_loader import load_config
from src.preprocessing import ben_graham_enhancement, crop_retinal_roi
from src.model import InceptionV3RetinopathyClassifier

config = load_config('../config/config.yaml')
print('Config loaded successfully!')
print('TensorFlow version:', tf.__version__)

## 1. Ben Graham Retinal Enhancement Comparison
Visualizing the effect of subtracting local average color (Gaussian blur) to amplify microaneurysms and hard exudates.

In [ ]:
from scripts.generate_synthetic_data import create_retinal_fundus_sample

# Generate sample Stage 3 (Severe) retinal image
raw_sample = np.array(create_retinal_fundus_sample(stage=3))
enhanced_sample = ben_graham_enhancement(raw_sample, target_size=(299, 299), sigma_x=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(raw_sample)
axes[0].set_title('Raw Fundus Photography', fontsize=12)
axes[0].axis('off')

axes[1].imshow(enhanced_sample)
axes[1].set_title('Ben Graham Enhanced (Normalized Illumination)', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. InceptionV3 Model Architecture Inspection
Building the InceptionV3 feature extractor with custom dual dense classification blocks.

In [ ]:
classifier = InceptionV3RetinopathyClassifier(config)
model = classifier.build()
model.summary()

## 3. Two-Phase Training Strategy
- **Phase 1**: Frozen InceptionV3 base, learning rate `1e-3`.
- **Phase 2**: Unfreeze from layer 249, learning rate `1e-5`.

In [ ]:
# Phase 1 Compilation
classifier.compile_for_phase1()
print('Phase 1 trainable weights:', len(model.trainable_weights))

# Phase 2 Compilation
classifier.compile_for_phase2()
print('Phase 2 trainable weights:', len(model.trainable_weights))